# pyfixest + MLflow tracking example

This notebook makes some synthetic data, runs it through `regress` (which wraps a pyfixest modeling function and logs to MLflow), and then pulls the logged runs back out to inspect them.

In [ ]:
import mlflow
import numpy as np
import pandas as pd
from IPython.display import HTML

from tracking import coefficients_table, regress, results_table

mlflow.set_tracking_uri("sqlite:///mlflow.db")
# Set the experiment once. regress then logs into the active experiment,
# so you don't pass name on every call.
mlflow.set_experiment("pyfixest-example")

## Make some data

A simple linear DGP: `Y = 2 + 1.5*X1 - 0.8*X2 + noise`.

In [ ]:
rng = np.random.default_rng(0)
n = 500

X1 = rng.normal(size=n)
X2 = rng.normal(size=n)
Y = 2.0 + 1.5 * X1 - 0.8 * X2 + rng.normal(scale=1.0, size=n)

data = pd.DataFrame({"Y": Y, "X1": X1, "X2": X2})
data.head()

## Run it

Two runs on the same data: one with `iid` errors, one with `hetero` (heteroskedasticity-robust) errors. The `vcov` is part of each run's content hash, so these are two distinct runs rather than one deduplicated run. Neither call passes `name` — they log into the experiment set above.

In [ ]:
fit_iid = regress("Y ~ X1 + X2", data=data, vcov="iid")
fit_hetero = regress("Y ~ X1 + X2", data=data, vcov="hetero")

fit_iid.summary()

## Get the runs data and show it

`regress` returns the fitted model directly, but everything it logged is also in MLflow. `results_table` wraps `mlflow.search_runs` and returns a tidy one-row-per-run frame (params and metrics, prefixes stripped).

In [ ]:
runs = results_table("pyfixest-example")
runs

## Artifacts logged per run

Each run also logs the coefficient table (`coefficients.json`) and a human-readable regression table (`summary.html`, rendered with pyfixest's `etable`).

In [ ]:
run_id = runs.loc[0, "run_id"]
[a.path for a in mlflow.artifacts.list_artifacts(run_id=run_id)]

## Render the logged regression table

The `summary.html` artifact is pyfixest's `etable` for that run. Load it back and render it inline.

In [ ]:
HTML(mlflow.artifacts.load_text(f"runs:/{run_id}/summary.html"))

## Coefficients across runs

`coefficients_table` reads each run's `coefficients.json` back into one long, self-describing frame (one row per run × coefficient, with the run's params joined on).

In [ ]:
coefficients_table("pyfixest-example")

Filter to a single coefficient to compare it across runs (here, the slope on `X1` under iid vs hetero errors):

In [ ]:
coefficients_table("pyfixest-example", coefficients="X1")